# Preprocess Voice and Extract Features

Convert `inputs/voice.webm` to mono 44.1 kHz WAV, extract high-frequency consonant texture, and analyze onset timing, RMS amplitude, and pYIN pitch.

In [1]:
from __future__ import annotations

import json
import shutil
import subprocess
from pathlib import Path

import imageio_ffmpeg
import librosa
import numpy as np
import soundfile as sf
from scipy.signal import butter, filtfilt

from voice_to_birdcall.paths import INPUTS_DIR, PROCESSED_DATA_DIR

SR = 44_100
CONSONANT_CUTOFF = 4_000
CONSONANT_ZCR_THRESHOLD = 0.1

voice_source = INPUTS_DIR / "voice.webm"
voice_path = PROCESSED_DATA_DIR / "voice.wav"
consonant_path = PROCESSED_DATA_DIR / "consonant.wav"
events_path = PROCESSED_DATA_DIR / "voice-events.json"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Convert Source Voice

Use FFmpeg to decode the WebM source and write `data/processed/voice.wav` as mono 44.1 kHz audio.

In [2]:
def convert_to_wav(source: Path, destination: Path) -> None:
    if not source.exists():
        raise FileNotFoundError(f"Voice source not found: {source}")

    ffmpeg = shutil.which("ffmpeg") or imageio_ffmpeg.get_ffmpeg_exe()

    command = [
        ffmpeg,
        "-y",
        "-i",
        str(source),
        "-ac",
        "1",
        "-ar",
        str(SR),
        str(destination),
    ]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("WAV conversion failed: " + result.stderr[-500:])


convert_to_wav(voice_source, voice_path)
print(f"Converted voice: {voice_path}")

Converted voice: C:\Users\morph\GitHub\voice-to-birdcall\data\processed\voice.wav


# Extract Consonant Texture

Keep high-energy frames with a zero-crossing rate above 0.1, isolate content above 4 kHz, and add short 15, 25, and 30 ms reflections.

In [3]:
def finite_audio(y: np.ndarray) -> np.ndarray:
    y = np.asarray(y, dtype=np.float32)
    if not np.all(np.isfinite(y)):
        y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
    return y


def normalize_audio(y: np.ndarray) -> np.ndarray:
    y = finite_audio(y)
    peak = float(np.max(np.abs(y))) if len(y) else 0.0
    return y if peak < 1e-8 else y / peak


def highpass_filter(y: np.ndarray, sr: int, cutoff: float) -> np.ndarray:
    if len(y) < 32:
        return y
    b, a = butter(4, cutoff / (sr / 2), btype="highpass")
    padlen = min(3 * max(len(a), len(b)), len(y) - 1)
    return filtfilt(b, a, y, padlen=padlen)


def load_audio(path: Path) -> np.ndarray:
    y, _ = librosa.load(str(path), sr=SR, mono=True)
    if len(y) == 0:
        raise ValueError(f"Audio file is empty: {path.name}")
    return normalize_audio(y)


def save_wav(path: Path, y: np.ndarray) -> None:
    sf.write(str(path), finite_audio(y), SR, subtype="PCM_16")


def extract_consonants(y: np.ndarray) -> np.ndarray:
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=2048, hop_length=512)[0]
    rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=512)[0]
    high_freq_y = highpass_filter(y, SR, cutoff=CONSONANT_CUTOFF)
    zcr_up = np.interp(np.linspace(0, 1, len(y)), np.linspace(0, 1, len(zcr)), zcr)
    rms_up = np.interp(np.linspace(0, 1, len(y)), np.linspace(0, 1, len(rms)), rms)
    threshold = np.percentile(rms_up, 65)
    mask = (zcr_up > CONSONANT_ZCR_THRESHOLD) & (rms_up > threshold)
    return high_freq_y * mask * 0.6


def apply_consonant_effects(part: np.ndarray) -> np.ndarray:
    if len(part) == 0 or np.max(np.abs(part)) < 1e-6:
        return part

    delayed = np.zeros_like(part)
    delay_30ms = int(0.03 * SR)
    if delay_30ms < len(part):
        delayed[delay_30ms:] = part[:-delay_30ms] * 0.4

    combined = part + delayed
    reverb = np.zeros_like(combined)
    feedback = 0.3
    for delay in [0.015, 0.025]:
        samples = int(delay * SR)
        if samples < len(combined):
            reverb[samples:] += combined[:-samples] * feedback
        feedback *= 0.7
    return (combined + reverb) * 0.5


voice = load_audio(voice_path)
consonant = apply_consonant_effects(extract_consonants(voice))
save_wav(consonant_path, consonant)

print(f"Voice duration: {len(voice) / SR:.2f} seconds")
print(f"Saved consonant texture to: {consonant_path}")

Voice duration: 3.72 seconds
Saved consonant texture to: C:\Users\morph\GitHub\voice-to-birdcall\data\processed\consonant.wav


# Analyze Voice Events

Detect onset times, interpolate RMS amplitude and pYIN pitch at each onset, and use the median voiced pitch as the relative pitch reference.

In [4]:
def get_voice_features(y: np.ndarray) -> tuple[list[dict], float]:
    onset_frames = librosa.onset.onset_detect(
        y=y,
        sr=SR,
        units="frames",
        backtrack=True,
        delta=0.08,
        wait=2,
    )
    onset_times = librosa.frames_to_time(onset_frames, sr=SR)
    rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=512)[0]
    rms_times = librosa.frames_to_time(np.arange(len(rms)), sr=SR, hop_length=512)
    f0, _, _ = librosa.pyin(
        y,
        fmin=librosa.note_to_hz("C3"),
        fmax=librosa.note_to_hz("C7"),
        sr=SR,
    )
    pitch_times = librosa.frames_to_time(np.arange(len(f0)), sr=SR)
    valid = ~np.isnan(f0)
    valid_pitch = f0[valid]
    median_pitch = float(np.median(valid_pitch)) if len(valid_pitch) else 440.0

    events = []
    for time in onset_times:
        amplitude = float(np.interp(time, rms_times, rms))
        pitch = float(
            np.interp(
                time,
                pitch_times[valid] if np.any(valid) else [0],
                f0[valid] if np.any(valid) else [median_pitch],
            )
        )
        events.append({"time": float(time), "amp": amplitude, "pitch": pitch})
    return events, median_pitch


events, median_pitch = get_voice_features(voice)
if not events:
    duration = len(voice) / SR
    events = [
        {"time": float(time), "amp": 0.7, "pitch": median_pitch}
        for time in np.arange(0.15, max(0.2, duration), 0.35)
    ]

analysis = {"sample_rate": SR, "median_pitch_hz": median_pitch, "events": events}
events_path.write_text(json.dumps(analysis, indent=2), encoding="utf-8")

print(f"Median pitch: {median_pitch:.1f} Hz")
print(f"Detected voice events: {len(events)}")
print(f"Saved voice features to: {events_path}")
events[:10]

Median pitch: 359.5 Hz
Detected voice events: 16
Saved voice features to: C:\Users\morph\GitHub\voice-to-birdcall\data\processed\voice-events.json


[{'time': 0.023219954648526078,
  'amp': 0.004420121666043997,
  'pitch': 365.74467430282664},
 {'time': 0.3250793650793651,
  'amp': 0.010920057073235512,
  'pitch': 365.74467430282664},
 {'time': 0.5688888888888889,
  'amp': 0.008428075350821018,
  'pitch': 365.74467430282664},
 {'time': 0.7662585034013606,
  'amp': 0.026105990633368492,
  'pitch': 365.74467430282664},
 {'time': 0.8591383219954648,
  'amp': 0.022848600521683693,
  'pitch': 365.74467430282664},
 {'time': 0.9171882086167801,
  'amp': 0.2928451895713806,
  'pitch': 361.5437384177477},
 {'time': 1.195827664399093,
  'amp': 0.025254415348172188,
  'pitch': 384.8813818384261},
 {'time': 1.4280272108843537,
  'amp': 0.14235322177410126,
  'pitch': 278.7883336231536},
 {'time': 1.6602267573696146,
  'amp': 0.08274157345294952,
  'pitch': 249.81097439559304},
 {'time': 2.147845804988662,
  'amp': 0.006337990052998066,
  'pitch': 382.8619182306804}]